In [0]:
# ==========================================================
# 03_silver_meta_brasil
# Camada: Silver
# Fonte: 2026-11-01_meta_alfabetizacao_brasil.parquet (Bronze)
# Destino: default.meta_brasil
#
# RESPONSABILIDADE
# Transformar a publicação oficial das metas de alfabetização
# (nível Brasil) em tabela Silver confiável.
#
# NOTA DE ARQUITETURA
# Esta é a única tabela de metas sem ambiguidade de fonte:
# existe apenas um arquivo de meta nacional na Bronze.
#
# NOTA SOBRE A COLUNA rede
# Nesta tabela rede é texto ('Pública'), não código numérico
# como no indicador. Não há join direto por rede — a Gold
# nacional usa a granularidade de ano.
#
# NOTA DE PERMISSÃO
# O schema silver pertence a outro usuário. A gravação ocorre
# em default, conforme adaptação documentada na seção 10.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA_DESTINO = "default"
TABELA_DESTINO = "meta_brasil"

BASE_BRONZE = (
    "/Workspace/Users/rm374983@fiap.com.br/"
    "postech-2iast-fase3-modelo-preditivo-g28/dados/bronze"
)
ARQUIVO_ORIGEM = "2026-11-01_meta_alfabetizacao_brasil.parquet"
CAMINHO_ORIGEM = f"{BASE_BRONZE}/{ARQUIVO_ORIGEM}"

TABELA_FULL = f"{CATALOGO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}"

print(f"Origem : {CAMINHO_ORIGEM}")
print(f"Destino: {TABELA_FULL}")

In [0]:
# ==========================================================
# Célula 2 — Leitura da Bronze
# ==========================================================

df_raw = spark.read.parquet(CAMINHO_ORIGEM)

print(f"Registros lidos: {df_raw.count():,}")
print(f"Colunas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# ==========================================================
# Célula 3 — Transformação e tipagem
#
# Decisões:
# - taxa_alfabetizacao e percentual_participacao são mantidas:
#   a taxa é o valor observado nacional, útil para a Gold
#   comparar meta vs realizado sem depender do indicador.
# - As 7 colunas de meta anual são mantidas como estão.
# - Dedupe por (ano, rede).
# ==========================================================

df_silver = (
    df_raw
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("rede").cast("string").alias("rede"),
        F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
        *[
            F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_alfabetizacao_{a}")
            for a in range(2024, 2031)
        ],
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
    )
    .filter(F.col("ano").isNotNull())
    .dropDuplicates(["ano", "rede"])
)

print(f"Registros após transformação: {df_silver.count():,}")

In [0]:
# ==========================================================
# Célula 4 — Rastreabilidade
# ingested_at, source e version em todos os registros.
# ==========================================================

df_silver = (
    df_silver
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit(ARQUIVO_ORIGEM))
    .withColumn("version", F.lit("v2"))
)

df_silver.show(truncate=False)

In [0]:
# ==========================================================
# Célula 5 — Data Quality
# Regras: completude, domínio e unicidade.
# Falhas críticas bloqueiam a gravação.
# ==========================================================

total = df_silver.count()

nulos_rede = df_silver.filter(F.col("rede").isNull()).count()
nulos_taxa = df_silver.filter(F.col("taxa_alfabetizacao").isNull()).count()

# Domínio: metas e taxa devem estar entre 0 e 100
metas = [f"meta_alfabetizacao_{a}" for a in range(2024, 2031)]
fora_faixa = df_silver.filter(
    F.greatest(*[
        F.when((F.col(m) < 0) | (F.col(m) > 100), 1).otherwise(0) for m in metas
    ]) == 1
).count()

chave = ["ano", "rede"]
duplicatas = total - df_silver.dropDuplicates(chave).count()

print("=" * 55)
print("DATA QUALITY — meta_brasil")
print("=" * 55)
print(f"Registros totais     : {total:,}")
print(f"Rede nula            : {nulos_rede:,}")
print(f"Taxa nula            : {nulos_taxa:,}")
print(f"Meta fora de 0-100   : {fora_faixa:,}")
print(f"Duplicatas na chave  : {duplicatas:,}")
print(f"Anos                 : {[r[0] for r in df_silver.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Redes                : {[r[0] for r in df_silver.select('rede').distinct().collect()]}")
print("=" * 55)

assert nulos_rede == 0, "FALHA CRÍTICA: rede nula"
assert nulos_taxa == 0, "FALHA CRÍTICA: taxa_alfabetizacao nula"
assert fora_faixa == 0, "FALHA CRÍTICA: meta fora do domínio 0-100"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 6 — Gravação da Silver
# ==========================================================

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FULL))

print(f"Gravado em {TABELA_FULL}")
print(f"Registros: {spark.table(TABELA_FULL).count():,}")

In [0]:
# ==========================================================
# Célula 7 — Validação pós-gravação
# ==========================================================

df_check = spark.table(TABELA_FULL)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")

print("\nConteúdo completo:")
df_check.select(
    "ano", "rede", "taxa_alfabetizacao",
    *[f"meta_alfabetizacao_{a}" for a in range(2024, 2031)],
    "percentual_participacao"
).orderBy("ano").show(truncate=False)